In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


df = pd.read_csv("housing.csv")

df["Income_Cat"] = pd.cut(df["median_income"],bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],labels=[1,2,3,4,5]).astype(int)


split = StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)

for train_index, test_index in split.split(df,df["Income_Cat"]) :
    
    train_data = df.loc[train_index]
    test_data = df.loc[test_index]
    

for sett in train_data,test_data :
    sett.drop(columns="Income_Cat",inplace = True,axis=1)

df = train_data.copy()

# Columns
categorical = ["ocean_proximity"]                          #list of categorical columns to be processed using the categorical pipeline.
features = df.drop(columns=categorical, axis=1).columns   #list of numerical columns to be processed using the numerical pipeline.

# Pipelines
Cat_Pipeline = Pipeline([
    ("Impute", SimpleImputer(strategy="most_frequent")),   #pipeline for categorical features, it imputes missing values with the most frequent value in each column.
    ("One_hot", OneHotEncoder(handle_unknown="ignore")),
])

Num_Pipeline = Pipeline([
    ("Impute", SimpleImputer(strategy="mean")),      #pipeline for numerical features, it imputes missing values with the mean value of each column.
    ("Min_Max", MinMaxScaler(feature_range=(-1,1))),
])

# ColumnTransformer
complete_Pipeline = ColumnTransformer([
    ("Num", Num_Pipeline, features),
    ("Cat", Cat_Pipeline, categorical),    ##Column Tranformer is used to join two pipelines together and apply them to the respective columns in the dataset.
])

df_prepared = complete_Pipeline.fit_transform(df)
cat_features = complete_Pipeline.named_transformers_["Cat"]["One_hot"].get_feature_names_out(categorical)
cat_features = [name.split("_", 2)[2] for name in cat_features]  
num_features = features
df_prepared = pd.DataFrame(df_prepared, columns= list(num_features)+list(cat_features) , index=df.index)


In [2]:
df_prepared

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
12655,-0.424303,0.270988,0.098039,-0.803276,-0.743879,-0.874772,-0.737117,-0.769148,-0.764533,0.0,1.0,0.0,0.0,0.0
15502,0.418327,-0.883103,-0.764706,-0.729664,-0.725193,-0.887217,-0.713966,-0.194852,0.091134,0.0,0.0,0.0,0.0,1.0
2908,0.057769,-0.398512,0.686275,-0.917994,-0.900773,-0.962779,-0.888723,-0.672405,-0.720822,0.0,1.0,0.0,0.0,0.0
14053,0.438247,-0.955367,-0.098039,-0.904818,-0.833441,-0.949830,-0.820388,-0.761865,-0.597936,0.0,0.0,0.0,0.0,1.0
20496,0.125498,-0.630181,0.019608,-0.820420,-0.792526,-0.897194,-0.784167,-0.448766,-0.079175,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15174,0.450199,-0.895855,-0.490196,-0.661240,-0.604059,-0.886600,-0.626960,-0.366891,0.045361,1.0,0.0,0.0,0.0,0.0
12661,-0.416335,0.268863,-0.450980,-0.598362,-0.542526,-0.732840,-0.471247,-0.680832,-0.689069,0.0,1.0,0.0,0.0,0.0
19263,-0.675299,0.253985,0.843137,-0.964338,-0.947165,-0.974495,-0.936520,-0.630378,-0.482885,1.0,0.0,0.0,0.0,0.0
19140,-0.671315,0.226355,-0.490196,-0.839803,-0.813789,-0.932453,-0.813667,-0.490145,0.002474,1.0,0.0,0.0,0.0,0.0
